In [2]:
!pip install xgboost


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 5.2 MB/s  0:00:39 eta 0:00:010:00:02


In [3]:
import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

train_df = pd.read_parquet('KDDTrain.parquet')
test_df = pd.read_parquet('KDDTest.parquet')

train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()

target_col = 'label' if 'label' in train_df.columns else 'class'
train_df['target'] = train_df[target_col].apply(lambda x: 0 if str(x).strip() == 'normal' else 1)
test_df['target'] = test_df[target_col].apply(lambda x: 0 if str(x).strip() == 'normal' else 1)

X_train_raw = train_df.drop(columns=[target_col, 'target'])
X_test_raw = test_df.drop(columns=[target_col, 'target'])
y_train = train_df['target']
y_test = test_df['target']

X_train = pd.get_dummies(X_train_raw, drop_first=True)
X_test = pd.get_dummies(X_test_raw, drop_first=True)
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

et_model = ExtraTreesClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
xgb_model = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss', n_jobs=-1)

et_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

print(f"Extra Trees Accuracy: {accuracy_score(y_test, et_model.predict(X_test))*100:.2f}%")
print(f"Random Forest Accuracy: {accuracy_score(y_test, rf_model.predict(X_test))*100:.2f}%")
print(f"XGBoost Accuracy: {accuracy_score(y_test, xgb_model.predict(X_test))*100:.2f}%")


Extra Trees Accuracy: 82.38%
Random Forest Accuracy: 83.18%
XGBoost Accuracy: 85.32%
